# WASP-69b cloud-free emission retrieval

This notebook runs the ROBERT cloud-free, one-region emission retrieval for the published WASP-69b spectrum. It treats each actual observing mode as an independent dataset exactly as supplied:

- JWST/NIRCam F322W2
- JWST/NIRCam F444W
- JWST/MIRI LRS

The six-point `Avg` product is excluded because it is a derived average of the NIRCam overlap region, not an additional instrument mode. There is no overlap spectrum, overlap forward model, overlap likelihood, stitching, offset, or observational rebinning. The total likelihood is simply the sum of one independent Gaussian likelihood per retained mode.

> This remains a validation analogue: it uses ROBERT's PG14 analytic temperature profile rather than the paper's full EGP-grid retrieval.

In [ ]:
from __future__ import annotations

import importlib.util
import json
import os
import sys
import time
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

def find_repo_root(start: Path | None = None) -> Path:
    candidate = (start or Path.cwd()).resolve()
    for directory in (candidate, *candidate.parents):
        if (directory / 'pyproject.toml').is_file() and (directory / 'src').is_dir():
            return directory
    raise RuntimeError('Could not find the ROBERT repository root')

ROOT = find_repo_root()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
print(f'Repository: {ROOT}')
print(f'Python: {sys.version.split()[0]} ({sys.executable})')

## Load the retrieval implementation

The numerical implementation is shared with the runnable example so fixes to the science path are picked up here rather than duplicated in notebook-only code.

In [ ]:
SCRIPT = ROOT / 'examples' / 'retrieve_wasp69b_nircam_cloud_free.py'
spec = importlib.util.spec_from_file_location('wasp69b_cloud_free_retrieval', SCRIPT)
if spec is None or spec.loader is None:
    raise RuntimeError(f'Cannot load {SCRIPT}')
workflow = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = workflow
spec.loader.exec_module(workflow)
print(f'Loaded retrieval implementation from {SCRIPT.relative_to(ROOT)}')

## Preserve native instrument modes

The loader is called with the MIRI offset disabled. The `Avg` overlap-average rows are then removed; every retained observation array and its published bin edges are otherwise passed through unchanged.

In [ ]:
from robert_exoplanets import ObservationCollection, load_schlawin2024_wasp69b

published = load_schlawin2024_wasp69b(
    ROOT / 'data' / 'wasp69b_schlawin2024',
    miri_offset_parameter=None,
)
RETAINED_MODES = ('f322w2', 'f444w', 'lrs')
observations = ObservationCollection(
    datasets=tuple(dataset for dataset in published.datasets if dataset.name in RETAINED_MODES),
    name='WASP-69b native instrument modes (no overlap-average product)',
    metadata={
        **dict(published.metadata),
        'selection': 'F322W2, F444W, and LRS; published Avg overlap product excluded',
        'overlap_handling': 'none',
    },
)

assert observations.names == RETAINED_MODES
assert all(dataset.offset_parameter is None for dataset in observations.datasets)
assert all(dataset.jitter_parameter is None for dataset in observations.datasets)
for dataset in observations.datasets:
    obs = dataset.observation
    print(
        f'{dataset.name:7s} | {obs.instrument:20s} | {obs.n_points:3d} points | '
        f'{obs.wavelength.min():.3f}-{obs.wavelength.max():.3f} micron'
    )
print(f'Total retained points: {observations.n_points}')

In [ ]:
colors = {'f322w2': '#20639b', 'f444w': '#ef5675', 'lrs': '#ffa600'}
fig, axis = plt.subplots(figsize=(10, 4.5))
for dataset in observations.datasets:
    obs = dataset.observation
    axis.errorbar(
        obs.wavelength, 1e6 * obs.flux, yerr=1e6 * obs.uncertainty,
        fmt='.', markersize=3, alpha=0.8, color=colors[dataset.name],
        label=f'{obs.instrument} ({obs.n_points} points)',
    )
axis.set(xlabel='Wavelength (micron)', ylabel='Eclipse depth (ppm)',
         title='WASP-69b: retained native observing modes')
axis.legend()
fig.tight_layout()

## Prepare opacity tables

This does **not** manipulate the observations. It integrates the high-resolution opacity tables onto each mode's published spectral bins so that each mode receives its own forward prediction. It can be slow on the first run; cached tables are reused after their source checksum is verified.

In [ ]:
PREPARE_OPACITIES = True  # Set True once if the per-mode cache does not exist.

if PREPARE_OPACITIES:
    started = time.monotonic()
    workflow.prepare_opacity_cache(observations)
    print(f'Opacity preparation completed in {(time.monotonic() - started) / 60:.1f} min')
else:
    missing = [
        workflow.CACHE / f'{dataset.name}_{species}.npz'
        for dataset in observations.datasets
        for species in workflow.SPECIES
        if not (workflow.CACHE / f'{dataset.name}_{species}.npz').is_file()
    ]
    print(f'{len(missing)} opacity cache file(s) missing')
    if missing:
        print('Set PREPARE_OPACITIES = True and rerun this cell before building the problem.')

## Build and inspect the retrieval problem

The atmosphere parameters are shared, but each retained mode has a separate forward spectrum and Gaussian term. No cross-mode covariance or overlap term is created.

In [ ]:
problem = workflow.build_problem(observations)
problem = replace(
    problem,
    name='wasp69b-cloud-free-native-modes',
    metadata={
        **dict(problem.metadata),
        'dataset_selection': 'F322W2,F444W,LRS',
        'difference': 'native F322W2/F444W/LRS modes and PG14 analytic TP instead of the full-band EGP RCE grid',
        'overlap_average': 'excluded',
        'dataset_manipulation': 'none',
    },
)
print(f'Problem: {problem.name}')
print(f'Datasets: {problem.observations.names}')
print(f'Parameters ({problem.ndim}): {problem.parameter_names}')
print(f'Likelihood: {problem.likelihood.name}')

## Preflight one forward model and likelihood

This catches opacity, chemistry, radiative-transfer, and shape errors before starting UltraNest. The printed terms make it explicit that only the three retained modes contribute.

In [ ]:
from robert_exoplanets import GaussianLikelihood

cube_midpoint = np.full(problem.ndim, 0.5)
theta = problem.prior_transform(cube_midpoint)
parameter_values = problem.parameter_mapping(theta)
started = time.monotonic()
prediction = problem.model_spectra(parameter_values)
elapsed = time.monotonic() - started
terms = {}
for dataset in problem.observations.datasets:
    mode_likelihood = GaussianLikelihood(
        include_normalization=problem.likelihood.include_normalization,
        offset_parameter=None,
        jitter_parameter=None,
        invalid_model_loglike=problem.likelihood.invalid_model_loglike,
        coordinate_rtol=problem.likelihood.coordinate_rtol,
        coordinate_atol=problem.likelihood.coordinate_atol,
    )
    terms[dataset.name] = mode_likelihood.loglike(
        prediction[dataset.name], dataset.observation, parameter_values
    )
for name in problem.observations.names:
    print(f'{name:7s}: spectrum={prediction[name].values.size:3d} points, logL={terms[name]:.3f}')
total_loglike = problem.likelihood.loglike(
    prediction,
    problem.observations,
    parameter_values,
)
assert np.isclose(sum(terms.values()), total_loglike)
print(f'Total logL: {total_loglike:.3f}')
print(f'Forward evaluation: {elapsed:.2f} s')

## Run or resume UltraNest

Run this cell repeatedly to extend the cumulative call ceiling in 5,000-call chunks. `show_status=True` provides live progress in the notebook. A single process is used because notebook kernels should not launch an MPI worker pool; use the Python script or a cluster job for MPI runs.

In [ ]:
from robert_exoplanets import run_ultranest

OUTPUT = ROOT / 'examples' / 'outputs' / 'wasp69b_cloud_free_native_modes'
LOG_DIR = OUTPUT / 'ultranest'
MAX_NCALLS = workflow._next_cumulative_call_limit(LOG_DIR)
print(f'UltraNest output: {LOG_DIR}')
print(f'Cumulative call ceiling for this attempt: {MAX_NCALLS}')

result = run_ultranest(
    problem,
    output_dir=LOG_DIR,
    min_num_live_points=50,
    max_ncalls=MAX_NCALLS,
    dlogz=2.5,
    resume='resume',
    show_status=True,
    mpi_nprocs=1,
    seed=20260711,
)

## Monitor status and inspect the best fit

The status cell can also be run while inspecting a resumed or previously interrupted output directory.

In [ ]:
status_path = LOG_DIR / 'sampler_status.json'
if status_path.is_file():
    print(json.dumps(json.loads(status_path.read_text()), indent=2))
else:
    print('No sampler status has been written yet.')

In [ ]:
best_parameters = result.best_fit_parameters
best_spectra = problem.model_spectra(best_parameters)
fig, (axis, residual_axis) = plt.subplots(
    2, 1, figsize=(10, 7), sharex=True, gridspec_kw={'height_ratios': [3, 1]}
)
for dataset in observations.datasets:
    name = dataset.name
    obs = dataset.observation
    model = best_spectra[name].values
    axis.errorbar(obs.wavelength, 1e6 * obs.flux, yerr=1e6 * obs.uncertainty,
                  fmt='.', markersize=3, color=colors[name], alpha=0.75, label=obs.instrument)
    axis.plot(obs.wavelength, 1e6 * model, color=colors[name], linewidth=1.3)
    residual_axis.plot(obs.wavelength, (obs.flux - model) / obs.uncertainty,
                       '.', markersize=3, color=colors[name])
axis.set_ylabel('Eclipse depth (ppm)')
axis.set_title('WASP-69b cloud-free emission: native-mode best fit')
axis.legend()
residual_axis.axhline(0, color='black', linewidth=0.8)
residual_axis.set(xlabel='Wavelength (micron)', ylabel='Residual / sigma')
fig.tight_layout()